# SnapUGC-LightKD Final 2000-Video Run

This notebook runs the clean v2 thesis pipeline on a larger Kaggle subset for a more stable validation estimate:

- DOVER-Mobile quality scoring
- CLIP/R(2+1)D/YAMNet/Sentence-T5/BLIP-base feature extraction
- Temporal Teacher, Student baseline, and Student+KD training


In [ ]:
# 0. Clone this repository into Kaggle working directory
GITHUB_REPO = "https://github.com/TranTop2806/SnapUGC-LightKD.git"
REPO_DIR = "/kaggle/working/SnapUGC-LightKD"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
    %cd /kaggle/working

print("Repo ready:", REPO_DIR)


In [ ]:
# 1. Install dependencies
!pip install -q -U "transformers>=4.49.0" accelerate sentence-transformers
!pip install -q eva-decord tensorflow tensorflow_hub scipy pandas tqdm matplotlib

import os, sys, glob, json, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Paths and ECR-balanced 2000-video subset
MAX_VIDEOS = 2000         # larger run for stable PLCC/SRCC estimates
SUBSET_SEED = 42          # deterministic ECR-stratified subset
ECR_BINS = 10             # sample roughly evenly across ECR quantiles
RUN_DOVER = True          # uses DOVER-Mobile for speed
RUN_CAPTION = True        # BLIP-base lightweight frame captions
RESET_FEATURES = True     # avoid resuming from a failed/empty previous run
CAPTION_DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

INPUT_DIR = '/kaggle/input/datasets/nguyntuncng/snapugc-dataset'
TRAIN_CSV_ORIG = os.path.join(INPUT_DIR, 'train_data.csv')
TRAIN_VIDEO_ROOT = os.path.join(INPUT_DIR, 'train_videos', 'train_videos')
if not os.path.exists(TRAIN_CSV_ORIG):
    raise FileNotFoundError(TRAIN_CSV_ORIG)
if not os.path.isdir(TRAIN_VIDEO_ROOT):
    raise FileNotFoundError(TRAIN_VIDEO_ROOT)

REPO_CANDIDATES = [
    globals().get('REPO_DIR', '/kaggle/working/SnapUGC-LightKD'),
    '/kaggle/working/SnapUGC-LightKD',
]
REPO_DIR = next((p for p in REPO_CANDIDATES if p and os.path.exists(os.path.join(p, 'scripts'))), None)
if REPO_DIR is None:
    raise FileNotFoundError('Cannot find SnapUGC-LightKD. Check the clone cell or update REPO_DIR.')

OUTPUT_DIR = f'/kaggle/working/final_{MAX_VIDEOS}_videos'
SUBSET_VIDEO_DIR = os.path.join(OUTPUT_DIR, 'subset_videos')
SUBSET_CSV = os.path.join(OUTPUT_DIR, f'train_subset_{MAX_VIDEOS}.csv')
FEATURES_PATH = os.path.join(OUTPUT_DIR, f'features_final_{MAX_VIDEOS}.json')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
DOVER_DIR = os.path.join(OUTPUT_DIR, 'dover')
DOVER_CSV = os.path.join(DOVER_DIR, 'dover_scores.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOVER_DIR, exist_ok=True)

# Build a deterministic ECR-stratified subset. Do not take the first N rows:
# the CSV is ordered and biases bounded runs toward very low ECR values.
import pandas as pd, subprocess
subset_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/make_subset.py'),
    '--csv', TRAIN_CSV_ORIG,
    '--videos', TRAIN_VIDEO_ROOT,
    '--out-csv', SUBSET_CSV,
    '--out-videos', SUBSET_VIDEO_DIR,
    '--max', str(MAX_VIDEOS),
    '--seed', str(SUBSET_SEED),
    '--bins', str(ECR_BINS),
]
if RESET_FEATURES:
    subset_cmd.append('--reset')
print(' '.join(subset_cmd))
subprocess.run(subset_cmd, check=True)

TRAIN_CSV = SUBSET_CSV
TRAIN_VIDEO_DIR = SUBSET_VIDEO_DIR
if RESET_FEATURES:
    for stale_path in [FEATURES_PATH, DOVER_CSV]:
        if os.path.exists(stale_path):
            os.remove(stale_path)

subset_df = pd.read_csv(TRAIN_CSV)
ecr = pd.to_numeric(subset_df['ECR'], errors='coerce')
print('INPUT_DIR:', INPUT_DIR)
print('TRAIN_CSV:', TRAIN_CSV)
print('TRAIN_VIDEO_DIR:', TRAIN_VIDEO_DIR)
print('Subset rows:', len(subset_df))
print('Subset videos:', len(glob.glob(os.path.join(TRAIN_VIDEO_DIR, '*.mp4'))))
print('Subset ECR:', {
    'min': float(ecr.min()),
    'max': float(ecr.max()),
    'mean': float(ecr.mean()),
    'std': float(ecr.std(ddof=0)),
})
print('Subset ECR quantile counts:')
print(pd.qcut(ecr, q=min(ECR_BINS, ecr.nunique(), len(ecr)), duplicates='drop').value_counts().sort_index())
print('REPO_DIR:', REPO_DIR)
print('FEATURES_PATH:', FEATURES_PATH)
print('CAPTION_DEVICE:', CAPTION_DEVICE)


In [ ]:
# 3. DOVER-Mobile quality scores for the bounded subset
# DOVER-Mobile is the official lightweight DOVER variant and is used for the full 2000-video subset.
if RUN_DOVER:
    !git clone -q https://github.com/VQAssessment/DOVER.git /kaggle/working/DOVER 2>/dev/null || true
    %cd /kaggle/working/DOVER
    !pip install -q scikit-video yacs timm einops opencv-python-headless decord pyyaml pandas scipy tqdm
    !mkdir -p pretrained_weights
    !wget -q -nc https://github.com/QualityAssessment/DOVER/releases/download/v0.5.0/DOVER-Mobile.pth -O pretrained_weights/DOVER-Mobile.pth
    !ls -lh pretrained_weights
    !python evaluate_a_set_of_videos.py -in "$TRAIN_VIDEO_DIR" -out "$DOVER_CSV" -o dover-mobile.yml
    print('DOVER_CSV:', DOVER_CSV)
    if not os.path.exists(DOVER_CSV):
        raise RuntimeError(f'DOVER did not produce expected CSV: {DOVER_CSV}')
    import pandas as pd
    dover_preview = pd.read_csv(DOVER_CSV)
    print('DOVER rows:', len(dover_preview))
    print(dover_preview.head())
    if len(dover_preview) < max(1, int(MAX_VIDEOS * 0.8)):
        raise RuntimeError(f'DOVER produced too few rows: {len(dover_preview)}/{MAX_VIDEOS}')
else:
    raise RuntimeError('RUN_DOVER must be True for this final bounded run.')

%cd /kaggle/working
print('DOVER_CSV:', DOVER_CSV)


In [ ]:
# 4. Feature extraction with BLIP-base captions
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/extract_features.py'),
    '--csv', TRAIN_CSV,
    '--videos', TRAIN_VIDEO_DIR,
    '--out', FEATURES_PATH,
    '--max', str(MAX_VIDEOS),
    '--save-every', '50',
    '--num-frames', '16',
    '--motion-clips', '4',
    '--motion-frames', '16',
    '--caption-frames', '3',
    '--caption-device', CAPTION_DEVICE,
    '--strict-caption',
    '--max-errors-before-abort', '1',
]
if DOVER_CSV:
    cmd += ['--dover-csv', DOVER_CSV]
if not RUN_CAPTION:
    cmd += ['--skip-caption']
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 5. Validate feature schema
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    features = json.load(f)
print('samples:', len(features))
if len(features) != MAX_VIDEOS:
    raise RuntimeError(f'Expected exactly {MAX_VIDEOS} successful samples, got {len(features)}.')

caption_ok = sum(1 for x in features if str(x.get('blip_caption') or x.get('caption') or '').strip())
dover_ok = sum(
    1 for x in features
    if bool((x.get('dover_scores') or {}).get('found', False))
)
print(f'BLIP non-empty captions: {caption_ok}/{len(features)}')
print(f'DOVER matched scores: {dover_ok}/{len(features)}')
if RUN_CAPTION and caption_ok != len(features):
    raise RuntimeError(f'Every sample must have a BLIP caption, got {caption_ok}/{len(features)}.')
if RUN_DOVER and dover_ok != len(features):
    raise RuntimeError(f'Every sample must have a matched DOVER score, got {dover_ok}/{len(features)}.')

first = features[0]
for key in ['clip_frame_embeddings', 'motion_clip_embeddings', 'dover_scores', 'yamnet_embedding_mean', 'metadata_text_embedding', 'caption_embedding', 'rationale_embedding', 'blip_caption']:
    value = first.get(key)
    if isinstance(value, list):
        print(key, 'len:', len(value), 'nested:', len(value[0]) if value and isinstance(value[0], list) else '')
    else:
        print(key, value if key in ['dover_scores', 'blip_caption'] else type(value))
print('First BLIP caption:', (first.get('blip_caption') or first.get('caption') or '')[:300])


In [ ]:
# 6. Train Teacher, Student baseline, and Student+KD
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/train.py'),
    '--data', FEATURES_PATH,
    '--save-dir', RESULTS_DIR,
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--teacher-epochs', '40',
    '--student-epochs', '60',
    '--batch', '16',
    '--teacher-hidden', '512',
    '--student-hidden', '256',
    '--teacher-blocks', '2',
    '--selection-metric', 'final_score',
    '--alpha', '0.5',
    '--beta', '0.3',
    '--attn-kd', '0.1',
    '--gamma', '0.2',
    '--delta', '0.2',
]
print(' '.join(cmd))
!{" ".join(cmd)}


In [ ]:
# 8. Show report and write diagnostics
report_path = os.path.join(RESULTS_DIR, 'final_experiment_report.json')
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)
print(json.dumps(report, indent=2)[:5000])

diag_cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/make_diagnostics.py'),
    '--features', FEATURES_PATH,
    '--report', report_path,
    '--subset-csv', TRAIN_CSV,
    '--out-dir', RESULTS_DIR,
    '--bins', str(ECR_BINS),
]
print(' '.join(diag_cmd))
subprocess.run(diag_cmd, check=True)
print('Diagnostics written:')
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, 'diagnostic_*')) + glob.glob(os.path.join(RESULTS_DIR, 'train_metrics.csv')) + glob.glob(os.path.join(RESULTS_DIR, 'feature_diagnostics.csv'))):
    print(path)


In [ ]:
# 9. Package outputs without subset videos
import zipfile
zip_path = f'/kaggle/working/snapugc_lightkd_final_{MAX_VIDEOS}_outputs.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

package_files = [SUBSET_CSV, FEATURES_PATH, DOVER_CSV]
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in package_files:
        if path and os.path.exists(path):
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))
    for root, _, files in os.walk(RESULTS_DIR):
        for name in files:
            path = os.path.join(root, name)
            zf.write(path, arcname=os.path.relpath(path, OUTPUT_DIR))

print(zip_path)
print('Excluded from zip:', SUBSET_VIDEO_DIR)
print('Zip size MB:', round(os.path.getsize(zip_path) / (1024 * 1024), 2))
